# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ARS-0/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

*Clones my repo on Colab and moves to the repo root, so the CSV path works.*

In [1]:
import os, sys, subprocess
import json
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ARS-0/FlyRank-ML-Internship"
REPO_DIR = "FlyRank-ML-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
os.makedirs("work/outputs", exist_ok=True)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Working dir:", os.getcwd())
print("Rows, columns:", df.shape)

Working dir: /content/FlyRank-ML-Internship
Rows, columns: (30000, 44)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
TIER_ORDER = ["top_3", "page_1", "striking", "page_3_5", "deep"]
MIN_N = 30            # a bucket with fewer than 30 pages is "too small to judge"
MIN_IMPR = 100        # pages under 100 impressions in 90 days are noise, not opportunities

# The ctr column in this file is in PERCENT (0.76 means 0.76%), so I rebuild CTR as
# clicks / impressions. Checking my unit assumption: the two should match.
check = (df["clicks_90d"] / df["impressions_90d"] * 100 - df["ctr"]).abs().max()
print(f"Biggest gap between ctr column (percent) and clicks/impressions*100: {check:.4f}")

d = df[df["impressions_90d"] >= MIN_IMPR].copy()
d["ctr_frac"] = d["clicks_90d"] / d["impressions_90d"]
print(f"Pages kept (impressions_90d >= {MIN_IMPR}): {len(d):,} of {len(df):,}")
print(f"Most clicks in any one page vs its impressions (should be <= 1): {d['ctr_frac'].max():.4f}")

# ---------- Signal 1 (flag-linked): CTR falls as position gets worse ----------
s1 = (d.groupby("position_tier")
        .agg(n=("ctr_frac", "size"),
             median_ctr_pct=("ctr_frac", lambda x: x.median() * 100),
             clicks=("clicks_90d", "sum"),
             impressions=("impressions_90d", "sum"))
        .loc[TIER_ORDER])
s1["pooled_ctr_pct"] = s1["clicks"] / s1["impressions"] * 100
print("\nSIGNAL 1: CTR by position tier (CTR in percent)")
print(s1[["n", "median_ctr_pct", "pooled_ctr_pct"]].round(3))
print("Pooled CTR never rises as tiers get worse:",
      bool((s1["pooled_ctr_pct"].diff().dropna() <= 0).all()))

# ---------- Signal 2: comparison articles vs keyword articles, inside each tier ----------
g = (d.groupby(["position_tier", "content_type"])
       .agg(n=("ctr_frac", "size"),
            median_ctr_pct=("ctr_frac", lambda x: x.median() * 100),
            clicks=("clicks_90d", "sum"),
            impressions=("impressions_90d", "sum"))
       .reset_index())
g["pooled_ctr_pct"] = g["clicks"] / g["impressions"] * 100
g["enough_pages"] = g["n"] >= MIN_N
g["position_tier"] = pd.Categorical(g["position_tier"], TIER_ORDER, ordered=True)
g = g.sort_values(["position_tier", "content_type"])
print("\nSIGNAL 2: CTR by content_type inside each tier (n printed, enough_pages = n >= 30)")
print(g[["position_tier", "content_type", "n", "median_ctr_pct", "pooled_ctr_pct", "enough_pages"]].round(3).to_string(index=False))

cmp_rows = []
for t in TIER_ORDER:
    c = g[(g.position_tier == t) & (g.content_type == "comparison article")]
    k = g[(g.position_tier == t) & (g.content_type == "keyword article")]
    if len(c) and len(k) and c.iloc[0]["enough_pages"] and k.iloc[0]["enough_pages"]:
        cmp_rows.append((t, bool(c.iloc[0]["pooled_ctr_pct"] < k.iloc[0]["pooled_ctr_pct"])))
print("\nTiers where BOTH comparison and keyword have >= 30 pages:", [t for t, _ in cmp_rows])
print("Comparison below keyword in:", sum(b for _, b in cmp_rows), "of", len(cmp_rows), "testable tiers")

Biggest gap between ctr column (percent) and clicks/impressions*100: 0.0050
Pages kept (impressions_90d >= 100): 22,006 of 30,000
Most clicks in any one page vs its impressions (should be <= 1): 0.1176

SIGNAL 1: CTR by position tier (CTR in percent)
                  n  median_ctr_pct  pooled_ctr_pct
position_tier                                      
top_3           533           0.187           0.487
page_1         8633           0.228           0.350
striking       5903           0.154           0.347
page_3_5       6058           0.062           0.155
deep            879           0.000           0.039
Pooled CTR never rises as tiers get worse: True

SIGNAL 2: CTR by content_type inside each tier (n printed, enough_pages = n >= 30)
position_tier       content_type    n  median_ctr_pct  pooled_ctr_pct  enough_pages
        top_3 comparison article    1           0.000           0.000         False
        top_3     feedly article    5           1.935           2.470         False
 

### Lane

I'm keeping **Lane 4: CTR / Engagement Opportunity Scoring.** The question is still which visible pages get fewer clicks than other pages at the same position.

### One correction from Week 1

In Notebook 01 I read some CTR values above 1.0 as "impossible." That was a units mistake. The `ctr` column is in **percent**, so 100 means 100%, and the file matches clicks / impressions × 100 exactly (the check above prints a gap near zero). The real problem with the "feedly article" numbers is the small sample: a few pages with a handful of impressions can show a huge CTR. I'm rebuilding CTR from clicks and impressions below so the units can't trip me up again.

### Signal 1 (linked to FlyRank's CTR-vs-position logic): **CONFIRMED**

Pages in better position tiers get a higher share of clicks. Pooled CTR is about 0.49% for top 3, 0.35% for page 1, 0.35% for striking distance, 0.15% for pages 3-5, and 0.04% for deep. It never goes up as position gets worse. Two caveats: page 1 and striking distance are nearly identical (0.350 vs 0.347), so those two tiers are not well separated; and by *median* CTR, top 3 (0.19%) sits a bit below page 1 (0.23%). The direction holds, but the tiers are not equally sharp. The rule only needs "better position, more clicks," so I'm keeping it.

### Signal 2 (content type inside a tier): **CONFIRMED, in the tiers I can test**

Comparison articles get a lower pooled CTR than keyword articles in all three tiers where both have at least 30 pages (page 1, striking, pages 3-5). I can't test top 3 (1 comparison page) or deep (0). Medians for comparison articles are 0 in every testable tier, meaning more than half of them got no clicks at all, so I lean on pooled CTR and call this a directional finding. "Feedly article" has under 30 pages in top 3 and deep, so I'm not drawing conclusions about it there.

### My rule, in plain words

A page is worth a title and meta description rewrite if it shows up a lot in search but gets fewer clicks than the typical page at its position tier. The score is **missed clicks**: (tier's click rate × the page's impressions) minus the page's actual clicks, never below zero. Only pages with at least 100 impressions in 90 days are scored.

- **Score:** missed clicks over the 90-day window
- **Reason code (only one):** `CTR_BELOW_TIER_PEERS`
- **Action label:** `REWRITE_TITLE_META`

Inputs used: `impressions_90d`, `clicks_90d`, `position_tier`. Nothing forward-looking, no product flags, no labels.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# tier "typical" click rate = total clicks / total impressions among scored pages in that tier
tier_rate = s1["clicks"] / s1["impressions"]

q = d.copy()
q["expected_ctr"] = q["position_tier"].map(tier_rate)
q["expected_clicks"] = q["expected_ctr"] * q["impressions_90d"]
q["score_missed_clicks"] = (q["expected_clicks"] - q["clicks_90d"]).clip(lower=0)
q["reason_code"] = np.where(q["score_missed_clicks"] > 0, "CTR_BELOW_TIER_PEERS", "NONE")
q["action_label"] = np.where(q["score_missed_clicks"] > 0, "REWRITE_TITLE_META", "NO_ACTION")

q = q.sort_values("score_missed_clicks", ascending=False).reset_index(drop=True)
q["rank"] = q.index + 1

out_cols = ["rank", "content_id", "client_id", "content_type", "position_tier", "avg_position",
            "impressions_90d", "clicks_90d", "expected_ctr", "expected_clicks",
            "score_missed_clicks", "reason_code", "action_label"]
q[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Queue written: work/outputs/baseline_action_score.csv  ({len(q):,} rows)")
print("Pages with a positive score (flagged):", int((q['score_missed_clicks'] > 0).sum()))
print(q["reason_code"].value_counts().to_string())

metrics = {
    "rows_in_file": int(len(df)),
    "rows_scored_impressions_ge_100": int(len(q)),
    "rows_flagged": int((q["score_missed_clicks"] > 0).sum()),
    "tier_pooled_ctr_pct": {k: round(float(v) * 100, 4) for k, v in tier_rate.items()},
    "signal_1_verdict": "CONFIRMED",
    "signal_2_verdict": "CONFIRMED (3 of 3 testable tiers)",
    "inputs_used": ["impressions_90d", "clicks_90d", "position_tier"],
}
with open("work/outputs/w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Metrics written: work/outputs/w04_baseline_metrics.json")

Queue written: work/outputs/baseline_action_score.csv  (22,006 rows)
Pages with a positive score (flagged): 15467
reason_code
CTR_BELOW_TIER_PEERS    15467
NONE                     6539
Metrics written: work/outputs/w04_baseline_metrics.json


## 3. Top-10 review

*For each of the top 10: the action, why it's there, and what would make it wrong.*

In [4]:
review_cols = ["rank", "content_id", "content_type", "position_tier", "avg_position",
               "impressions_90d", "clicks_90d", "expected_clicks", "score_missed_clicks"]
top10 = q.head(10).copy()
top10["actual_ctr_pct"] = (top10["clicks_90d"] / top10["impressions_90d"] * 100).round(3)
top10["tier_ctr_pct"] = (top10["expected_ctr"] * 100).round(3)
print(top10[review_cols + ["actual_ctr_pct", "tier_ctr_pct"]].round(3).to_string(index=False))

# extra context for the "what would make it wrong" lines
ctx = df.set_index("content_id").loc[top10["content_id"],
        ["impressions_last_30d", "impressions_prev_30d", "content_age_days", "main_intent"]]
print()
print(ctx.to_string())

 rank           content_id    content_type position_tier  avg_position  impressions_90d  clicks_90d  expected_clicks  score_missed_clicks  actual_ctr_pct  tier_ctr_pct
    1 content_8c19996aa890 keyword article         top_3           2.5           509252         785         2480.737             1695.737           0.154         0.487
    2 content_8451fc6f034d keyword article         top_3           2.3           272144          75         1325.704             1250.704           0.028         0.487
    3 content_5fe46e04994d keyword article        page_1           4.2           517715         741         1811.247             1070.247           0.143         0.350
    4 content_36ff89c8214e keyword article        page_1           7.3           295097         154         1032.409              878.409           0.052         0.350
    5 content_c8e9d6ab9013 keyword article        page_1           9.7           208678           0          730.068              730.068           0.000       

All ten: **action = `REWRITE_TITLE_META`**, **reason code = `CTR_BELOW_TIER_PEERS`**. Numbers come from the table above.

1. **`content_8c19996aa890`** (top 3, 509k impressions, 785 clicks vs about 2,480 expected). Biggest gap in the queue. *Wrong if:* the question is answered right in the search result, so people don't need to click.
2. **`content_8451fc6f034d`** (top 3, 272k impressions, 75 clicks, 0.03% vs 0.49% for the tier). *Wrong if:* the page ranks for a few broad queries that pile up impressions without real interest. Impressions also jumped recently (169k last 30 days vs 98k the 30 before), and a 90-day average can hide that.
3. **`content_5fe46e04994d`** (page 1, avg position 4.2, 518k impressions, 741 clicks). *Wrong if:* the average position hides a mix, like #1 for one query and #9 for another.
4. **`content_36ff89c8214e`** (page 1, avg position 7.3, 295k impressions, 154 clicks). *Wrong if:* most impressions come from one search where a feature (video, box, ads) sits above it.
5. **`content_c8e9d6ab9013`** (page 1, 208k impressions, **0 clicks**). *Wrong if:* click tracking is broken for this page. Zero clicks in 88 days of impressions looks more like a data problem than a title problem, so check the data before touching the title.
6. **`content_c84a0ab98e90`** (page 1, 223k impressions, 70 clicks, 95 days old). *Wrong if:* it's a young page whose position moved a lot, so the average doesn't describe where it sat most of the time.
7. **`content_e12868d1f396`** (top 3, avg position 2.9, 150k impressions, 104 clicks). *Wrong if:* "2.9" is really a mix of #1 and #6 and the tier benchmark is too generous.
8. **`content_4a6607efcb46`** (top 3, 128k impressions, 17 clicks). *Wrong if:* the recent surge is the cause. Last-30-day impressions are 122k against about 2k the month before, so nearly all impressions are new and the 90-day CTR mostly reflects one odd month. Might be a new query, not a bad title.
9. **`content_cb112fce36be`** (page 1, 310k impressions, 492 clicks, transactional intent). *Wrong if:* searchers with buying intent click ads or shopping results above the organic listing, so a lower organic CTR is normal.
10. **`content_aaef01a50def`** (page 1, 517k impressions, 1,270 clicks, 0.25% vs 0.35%). The weakest of the ten: only about 30% below peers. *Wrong if:* that gap is inside the normal spread for page-1 pages. It ranks high only because of its sheer impressions.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
# 1) Leakage check: only these columns fed the score
used = ["impressions_90d", "clicks_90d", "position_tier"]
banned = ["health_score", "priority_score", "action_type"]
print("Columns used by the rule:", used)
print("Banned columns present in the file at all:", [c for c in banned if c in df.columns])
assert not any(c in used for c in banned)
print("No product flags and no future-window columns (_last_30d / _prev_30d) went into the score.")

# 2) Where do the content types sit in the queue? (Signal 2 said comparison articles underperform.)
share = (q.assign(top500=q["rank"] <= 500)
          .groupby("content_type")
          .agg(pages=("rank", "size"), in_top_500=("top500", "sum"), median_rank=("rank", "median")))
print("\nQueue position by content type")
print(share)

# 3) How concentrated is the top of the queue?
print("\nClients in the top 10:", q.head(10)["client_id"].nunique(), "| top 100:", q.head(100)["client_id"].nunique())
print("Total score held by the top 10 pages / all pages:",
      round(q.head(10)["score_missed_clicks"].sum() / q["score_missed_clicks"].sum() * 100, 2), "%")

Columns used by the rule: ['impressions_90d', 'clicks_90d', 'position_tier']
Banned columns present in the file at all: []
No product flags and no future-window columns (_last_30d / _prev_30d) went into the score.

Queue position by content type
                    pages  in_top_500  median_rank
content_type                                      
comparison article    366           0      12681.0
feedly article        352           0      14353.5
keyword article     21288         500      10835.5

Clients in the top 10: 5 | top 100: 10
Total score held by the top 10 pages / all pages: 5.38 %


### Weak picks

- **#5 (`content_c8e9d6ab9013`)** is the one I trust least: 208k impressions and zero clicks smells like broken tracking, not a bad title. A real analyst would check the data first.
- **#10** is barely below its peers and only ranks because it has huge impressions.
- **#8** is mostly one recent surge in impressions, so its 90-day CTR is a poor summary.
- **Size bias:** the score is missed *clicks*, so a high-impression page always wins over a smaller page with a worse CTR. That's on purpose (more clicks to win back), but it means small pages with terrible CTRs never show up near the top. The table above shows where comparison articles land: they have a lower CTR (Signal 2), but few impressions, so none of them land in the top 500 (see the table above).
- **Benchmark:** each tier's click rate is computed from the same pages I score, and the 90-day window is the only one used. It's a baseline to beat, not a tested prediction.

### Leakage check

The rule uses only `impressions_90d`, `clicks_90d`, and `position_tier`. No product flags, no label, no `_last_30d` or `_prev_30d` columns. I used careful words on purpose: these are observed, directional numbers for decision support, not proof of cause.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.